# Assumptions and Scope

## Assumptions

- This notebook is for educational and demonstration purposes only and is not production-ready.
- The implementation is framework-free no LangChain or similar orchestration frameworks are used.
- The system uses public APIs without authentication (no API keys).
- The LLM is local Mistral.
- Tool input validation is minimal, outputs from the LLM are not strictly enforced against schemas.
- Error handling is simplified and does not cover all edge cases.
- No security mechanisms are implemented (e.g., prompt injection protection, sanitization).

## In Scope

- Demonstration of an LLM-driven agent loop:
  - reasoning → tool selection → execution → response
- Use of a MCP server
- Tool integrations:
  - current weather (Open-Meteo)
  - latest news (RSS feeds)
- Basic working memory for maintaining conversational context
- Iterative reasoning with a configurable max iteration limit

## Out of Scope

- Production-grade capabilities:
  - authentication / authorization
  - API key management
  - retries, backoff, circuit breakers
- Asynchronous or streaming execution
- Multi-agent or distributed MCP architecture
- Reflection on an output
- Persistent storage or long-term memory
- Monitoring, tracing, observability
- Security protections (prompt injection, tool misuse)
- UI / frontend integration

## Scenario

The system answers user questions about weather and news using an LLM for reasoning and tool selection.

### Flow

1. The user submits a natural language query  
2. The agent:
   - analyzes intent using the LLM  
   - determines whether a tool is required  
3. If needed, the agent:
   - selects the appropriate tool  
   - invokes it via the MCP server  
4. The tool returns structured data  
5. The LLM generates a final natural language response  

## Architecture Overview

This notebook implements a modular agent system that mimics the Model Context Protocol (MCP) architecture.

1. **MCP Server Abstraction Layer**: Independent tool servers with standardized interfaces
2. **Tool Schema**: Strict JSON schema for tool definitions and validation
3. **Agent**: LLM-driven decision making with tool execution loop
4. **Working Memory**: Efficient context management without full history
5. **Evaluation Framework**: Metrics for agent performance

## 1. Environment Setup

Pydantic is a data validation and parsing library that uses Python type hints to enforce structure. Typing-extensions provides new typing features that are not yet available in your Python version.

In [ ]:
pip install requests feedparser ollama pydantic typing-extensions
ollama pull mistral

In [1]:
import json
from abc import ABC, abstractmethod
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum

import requests
import feedparser
import ollama
from pydantic import BaseModel, Field, ValidationError

## 2. MCP Server Abstraction Layer

The base abstraction defines the contract for MCP server.

In [2]:
class ToolSchema(BaseModel):
    """Strict tool schema following MCP specification."""
    name: str = Field(..., description="Unique tool identifier")
    description: str = Field(..., description="Clear description of tool purpose")
    input_schema: Dict[str, Any] = Field(..., description="JSON Schema for tool arguments")


class ToolResult(BaseModel):
    """Standardized tool execution result."""
    success: bool
    data: Optional[Any] = None
    error: Optional[str] = None


class MCPServer(ABC):
    """Abstract base class for all MCP servers.
    
    Each server must implement tool listing and execution.
    This abstraction enables:
    - Dynamic tool discovery
    - Standardized tool calling interface
    - Easy extensibility with new servers
    """
    
    def __init__(self, name: str):
        self.name = name
        self._tools: Dict[str, ToolSchema] = {}
        self._register_tools()
    
    @abstractmethod
    def _register_tools(self) -> None:
        """Register all tools provided by this server."""
        pass
    
    def list_tools(self) -> List[ToolSchema]:
        """Return all available tools from this server."""
        return list(self._tools.values())
    
    def call_tool(self, name: str, arguments: Dict[str, Any]) -> ToolResult:
        """Execute a tool by name with provided arguments.
        
        Args:
            name: Tool identifier
            arguments: Tool parameters
            
        Returns:
            ToolResult with success status and data/error
        """
        if name not in self._tools:
            return ToolResult(
                success=False,
                error=f"Tool '{name}' not found in server '{self.name}'"
            )
        
        try:
            method_name = f"_execute_{name}"
            if not hasattr(self, method_name):
                return ToolResult(
                    success=False,
                    error=f"Implementation for '{name}' not found"
                )
            
            result = getattr(self, method_name)(arguments)
            return ToolResult(success=True, data=result)
        
        except Exception as e:
            print(f"Tool execution failed: {name}", exc_info=True)
            return ToolResult(success=False, error=str(e))
    
    def _add_tool(self, schema: ToolSchema) -> None:
        """Internal method to register a tool schema."""
        self._tools[schema.name] = schema

## 3. MCP Server Implementation

Implements weather data retrieval using Open-Meteo (no API key required) and news retrieval using RSS feeds (no API key required).

In [4]:
class MCPServerImpl(MCPServer):
    """MCP Server for weather and news data.
    
    Provides tools for:
    - Current weather conditions (Open-Meteo API)
    - Latest news headlines (RSS feeds)
    
    """
    
    # Weather API endpoints
    GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
    WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
    
    # News RSS feeds by category
    RSS_FEEDS = {
        "general": "http://rss.cnn.com/rss/edition.rss",
        "technology": "http://rss.cnn.com/rss/edition_technology.rss",
        "business": "http://rss.cnn.com/rss/edition_business.rss",
        "world": "http://rss.cnn.com/rss/edition_world.rss"
    }
    
    def __init__(self):
        super().__init__("unified")
    
    def _register_tools(self) -> None:
        """Register all tools (weather + news)."""
        self._add_tool(ToolSchema(
            name="get_current_weather",
            description="Get current weather conditions for a specific location. Returns temperature, conditions, wind speed, and humidity.",
            input_schema={
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name or location (e.g., 'London', 'New York')"
                    }
                },
                "required": ["location"]
            }
        ))
        
        self._add_tool(ToolSchema(
            name="get_latest_news",
            description="Get the latest news headlines. Can filter by category (general, technology, business, world). Returns top news stories with titles and summaries.",
            input_schema={
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "description": "News category: general, technology, business, or world",
                        "enum": ["general", "technology", "business", "world"]
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Maximum number of articles to return (default: 5)",
                        "default": 5
                    }
                },
                "required": ["category"]
            }
        ))
    
    # ========== Weather Tool Implementation ==========
    
    def _get_coordinates(self, location: str) -> Tuple[float, float, str]:
        """Convert location name to coordinates using geocoding.
        
        Returns:
            Tuple of (latitude, longitude, full_location_name)
        """
        response = requests.get(
            self.GEOCODING_URL,
            params={"name": location, "count": 1, "language": "en", "format": "json"},
            timeout=10
        )
        response.raise_for_status()
        
        data = response.json()
        if not data.get("results"):
            raise ValueError(f"Location not found: {location}")
        
        result = data["results"][0]
        return (
            result["latitude"],
            result["longitude"],
            f"{result['name']}, {result.get('country', '')}"
        )
    
    def _execute_get_current_weather(self, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """Implementation of get_current_weather tool."""
        location = arguments["location"]
        
        # Get coordinates for location
        lat, lon, full_location = self._get_coordinates(location)
        
        # Fetch weather data
        response = requests.get(
            self.WEATHER_URL,
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
                "temperature_unit": "celsius",
                "wind_speed_unit": "kmh"
            },
            timeout=10
        )
        response.raise_for_status()
        
        data = response.json()
        current = data["current"]
        
        # Map weather codes to conditions
        weather_code = current["weather_code"]
        conditions = self._get_weather_description(weather_code)
        
        return {
            "location": full_location,
            "temperature_celsius": current["temperature_2m"],
            "conditions": conditions,
            "humidity_percent": current["relative_humidity_2m"],
            "wind_speed_kmh": current["wind_speed_10m"],
            "timestamp": current["time"]
        }
    
    def _get_weather_description(self, code: int) -> str:
        """Map WMO weather code to description."""
        code_map = {
            0: "Clear sky",
            1: "Mainly clear",
            2: "Partly cloudy",
            3: "Overcast",
            45: "Foggy",
            48: "Foggy",
            51: "Light drizzle",
            53: "Moderate drizzle",
            55: "Dense drizzle",
            61: "Slight rain",
            63: "Moderate rain",
            65: "Heavy rain",
            71: "Slight snow",
            73: "Moderate snow",
            75: "Heavy snow",
            95: "Thunderstorm"
        }
        return code_map.get(code, "Unknown")
    
    # ========== News Tool Implementation ==========
    
    def _execute_get_latest_news(self, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """Implementation of get_latest_news tool."""
        category = arguments["category"]
        limit = arguments.get("limit", 5)
        
        if category not in self.RSS_FEEDS:
            raise ValueError(f"Unknown category: {category}")
        
        # Fetch RSS feed
        feed_url = self.RSS_FEEDS[category]
        feed = feedparser.parse(feed_url)
        
        if feed.bozo:
            raise ValueError(f"Failed to parse RSS feed: {feed_url}")
        
        # Extract articles
        articles = []
        for entry in feed.entries[:limit]:
            articles.append({
                "title": entry.get("title", "No title"),
                "summary": entry.get("summary", "No summary"),
                "link": entry.get("link", ""),
                "published": entry.get("published", "")
            })
        
        return {
            "category": category,
            "count": len(articles),
            "articles": articles,
            "retrieved_at": datetime.now().isoformat()
        }

## 4. Working Memory

Efficient context management that maintains only essential information.

- Maintains a rolling summary of conversation context
- Keeps only last N messages for immediate context
- Stores tool results separately for reference
- Reduces token usage while maintaining coherence

In [5]:
@dataclass
class Message:
    """Represents a single message in the conversation."""
    role: str  # 'user', 'assistant', 'tool'
    content: str
    timestamp: datetime = field(default_factory=datetime.now)


class WorkingMemory:
    """Efficient working memory that avoids sending full chat history."""
    
    def __init__(self, max_recent_messages: int = 3):
        self.max_recent_messages = max_recent_messages
        self.summary: str = ""
        self.recent_messages: List[Message] = []
        self.tool_results: Dict[str, Any] = {}  # tool_name -> last_result
    
    def add_message(self, role: str, content: str) -> None:
        """Add a message to recent history."""
        message = Message(role=role, content=content)
        self.recent_messages.append(message)
        
        # Trim to max size
        if len(self.recent_messages) > self.max_recent_messages:
            # Move oldest to summary if needed
            self.recent_messages = self.recent_messages[-self.max_recent_messages:]
    
    def add_tool_result(self, tool_name: str, result: Any) -> None:
        """Store tool execution result."""
        self.tool_results[tool_name] = result
    
    def update_summary(self, summary: str) -> None:
        """Update conversation summary."""
        self.summary = summary
    
    def get_context(self) -> List[Dict[str, str]]:
        """Build context for LLM with summary and recent messages.
        
        Returns:
            List of message dicts for LLM consumption
        """
        context = []
        
        # Add summary if exists
        if self.summary:
            context.append({
                "role": "system",
                "content": f"Conversation summary: {self.summary}"
            })
        
        # Add recent messages
        for msg in self.recent_messages:
            context.append({
                "role": msg.role,
                "content": msg.content
            })
        
        return context
    
    def clear(self) -> None:
        """Reset working memory."""
        self.summary = ""
        self.recent_messages = []
        self.tool_results = {}

## 5. Agent

Core agent loop with LLM-driven tool selection and execution. 

Architecture:
1. Discovers tools from all MCP servers
2. Constructs system prompt with tool schemas
3. Processes user query through LLM
4. Parses tool calls from LLM response
5. Executes tools via MCP servers
6. Feeds results back to LLM
7. Returns final answer
    
Agent Loop:
1. Add user query to memory
2. Build context from working memory
3. Call LLM with system prompt + context
4. Parse response for tool calls
5. If tool call found: execute and loop back to step 2
6. If no tool call: return final answer
7. Enforce max iterations to prevent infinite loops


In [10]:
class ToolCall(BaseModel):
    """Parsed tool call from LLM response."""
    name: str
    arguments: Dict[str, Any]


class AgentOrchestrator:
    """Agent orchestrator with LLM-driven tool selection."""
    
    def __init__(
        self,
        mcp_server: MCPServer,
        model: str = "mistral",
        max_iterations: int = 5
    ):
        self.mcp_server = mcp_server
        self.model = model
        self.max_iterations = max_iterations
        self.memory = WorkingMemory()
        
        self.tools: Dict[str, ToolSchema] = {}
        self._discover_tools()
    
    def _discover_tools(self) -> None:
        """Discover all available tools from MCP servers."""
        for tool in self.mcp_server.list_tools():
            self.tools[tool.name] = tool
    
    def _build_system_prompt(self) -> str:
        """Build system prompt with tool schemas."""
        tool_descriptions = []
        for tool_name, tool_schema in self.tools.items():
            tool_descriptions.append(
                f"- {tool_schema.name}: {tool_schema.description}\n"
                f"  Parameters: {json.dumps(tool_schema.input_schema, indent=2)}"
            )
        
        tools_text = "\n\n".join(tool_descriptions)
        
        return f"""You are a helpful AI assistant with access to tools for answering questions.

AVAILABLE TOOLS:
{tools_text}

INSTRUCTIONS:
1. Analyze the user's question carefully
2. Determine if you need to use a tool to answer
3. If you need a tool, respond with a JSON tool call in this EXACT format:
   <TOOL_CALL>
   {{"name": "tool_name", "arguments": {{"param": "value"}}}}
   </TOOL_CALL>
4. If you don't need a tool, or after receiving tool results, provide a natural language answer
5. Be concise and accurate
6. Do not hallucinate information - only use data from tool results

IMPORTANT:
- Output ONLY valid JSON inside <TOOL_CALL>
- Do NOT include any extra text inside TOOL_CALL
- Ensure JSON is strictly valid (double quotes, no trailing commas)
- Only make ONE tool call at a time
- Wait for tool results before answering
- Never invent tool results
- If unsure, ask for clarification
"""
    
    def _parse_tool_call(self, response: str) -> Optional[ToolCall]:
        """Parse tool call from LLM response. Returns ToolCall object if found, None otherwise"""
        try:
            # Extract JSON between markers
            if "<TOOL_CALL>" in response and "</TOOL_CALL>" in response:
                start = response.index("<TOOL_CALL>") + len("<TOOL_CALL>")
                end = response.index("</TOOL_CALL>")
                json_str = response[start:end].strip()
                
                data = json.loads(json_str)
                return ToolCall(**data)
        except (ValueError, json.JSONDecodeError, ValidationError) as e:
            print(f"Failed to parse tool call: {e}")
        
        return None
    
    def _execute_tool(self, tool_call: ToolCall) -> ToolResult:
        """Execute tool via MCP server."""
        if tool_call.name not in self.tools:
            return ToolResult(
                success=False,
                error=f"Unknown tool: {tool_call.name}"
            )
        
        result = self.mcp_server.call_tool(tool_call.name, tool_call.arguments)
        
        # Store in memory
        if result.success:
            self.memory.add_tool_result(tool_call.name, result.data)
        
        return result
    
    def _call_llm(self, messages: List[Dict[str, str]]) -> str:
        """Call LLM.
        
        Args:
            messages: List of message dicts with role and content
            
        Returns:
            LLM response text
        """
        try:
            response = ollama.chat(
                model=self.model,
                messages=messages,
                options={
                    "temperature": 0.0,
                    "top_p": 0.9,
                }
            )
            return response['message']['content']
        except Exception as e:
            print(f"LLM call failed: {e}", exc_info=True)
            raise
    
    def process_query(self, query: str) -> str:
        """Process user query through agent loop."""
        
        # Add query to memory
        self.memory.add_message("user", query)
        
        system_prompt = self._build_system_prompt()
        
        for iteration in range(self.max_iterations):
            # Build messages for LLM
            messages = [
                {"role": "system", "content": system_prompt}
            ] + self.memory.get_context()
            
            # Call LLM
            response = self._call_llm(messages)
            print(f"LLM response: {response[:200]}...")
            
            # Parse for tool call
            tool_call = self._parse_tool_call(response)
            
            if tool_call is None:
                self.memory.add_message("assistant", response)
                return response
            
            # Execute tool
            tool_result = self._execute_tool(tool_call)
            
            # Add tool result to memory
            if tool_result.success:
                result_text = f"Tool '{tool_call.name}' returned: {json.dumps(tool_result.data, indent=2)}"
            else:
                result_text = f"Tool '{tool_call.name}' failed: {tool_result.error}"
            
            self.memory.add_message("tool", result_text)
            print(f"Tool result: {result_text[:200]}...")
        
        # Max iterations reached
        return "I apologize, but I couldn't complete your request within the allowed iterations."
    
    def reset(self) -> None:
        """Reset agent state."""
        self.memory.clear()

## 6. Evaluation Framework

Quantitative metrics for measuring agent performance.

In [7]:
@dataclass
class EvaluationQuery:
    """Single evaluation test case."""
    query: str
    expected_tool: str
    category: str


@dataclass
class EvaluationResult:
    """Result of evaluating a single query."""
    query: str
    expected_tool: str
    actual_tool: Optional[str]
    tool_correct: bool
    response: str
    error: Optional[str] = None


class AgentEvaluator:
    """Evaluation framework for agent performance.
    
    Metrics:
    1. Tool Selection Accuracy: Did agent call correct tool?
    2. Success Rate: Did agent complete without errors?
    """
    
    # Evaluation dataset
    EVAL_QUERIES = [
        EvaluationQuery(
            query="What's the weather like in London?",
            expected_tool="get_current_weather",
            category="weather"
        ),
        EvaluationQuery(
            query="Tell me the temperature in Tokyo",
            expected_tool="get_current_weather",
            category="weather"
        ),
        EvaluationQuery(
            query="What are the latest technology news?",
            expected_tool="get_latest_news",
            category="news"
        ),
        EvaluationQuery(
            query="Show me recent business headlines",
            expected_tool="get_latest_news",
            category="news"
        ),
        EvaluationQuery(
            query="What's happening in the world today?",
            expected_tool="get_latest_news",
            category="news"
        ),
        EvaluationQuery(
            query="How's the weather in New York?",
            expected_tool="get_current_weather",
            category="weather"
        ),
    ]
    
    def __init__(self, agent: AgentOrchestrator):
        self.agent = agent
    
    def _extract_tool_used(self, memory: WorkingMemory) -> Optional[str]:
        """Extract which tool was used from memory.
        
        Returns:
            Tool name if found in tool_results, None otherwise
        """
        if self.agent.memory.tool_results:
            # Return first tool used (most queries use only one)
            return list(self.agent.memory.tool_results.keys())[0]
        return None
    
    def evaluate_single(self, eval_query: EvaluationQuery) -> EvaluationResult:
        """Evaluate a single query.
        
        Args:
            eval_query: Query to evaluate
            
        Returns:
            EvaluationResult with metrics
        """
        try:
            # Reset agent state
            self.agent.reset()
            
            # Process query
            response = self.agent.process_query(eval_query.query)
            
            # Extract tool used
            actual_tool = self._extract_tool_used(self.agent.memory)
            
            # Check if correct tool was used
            tool_correct = actual_tool == eval_query.expected_tool
            
            return EvaluationResult(
                query=eval_query.query,
                expected_tool=eval_query.expected_tool,
                actual_tool=actual_tool,
                tool_correct=tool_correct,
                response=response
            )
        
        except Exception as e:
            print(f"Evaluation failed for query: {eval_query.query}", exc_info=True)
            return EvaluationResult(
                query=eval_query.query,
                expected_tool=eval_query.expected_tool,
                actual_tool=None,
                tool_correct=False,
                response="",
                error=str(e)
            )
    
    def evaluate_all(self) -> Dict[str, Any]:
        """Run full evaluation suite.
        
        Returns:
            Dict with overall metrics and individual results
        """
        
        results = []
        for eval_query in self.EVAL_QUERIES:
            print(f"Evaluating: {eval_query.query}")
            result = self.evaluate_single(eval_query)
            results.append(result)
        
        # Calculate metrics
        total = len(results)
        tool_correct_count = sum(1 for r in results if r.tool_correct)
        success_count = sum(1 for r in results if r.error is None)
        
        tool_accuracy = tool_correct_count / total if total > 0 else 0
        success_rate = success_count / total if total > 0 else 0
        
        return {
            "total_queries": total,
            "tool_selection_accuracy": tool_accuracy,
            "success_rate": success_rate,
            "results": results
        }
    
    def print_report(self, evaluation: Dict[str, Any]) -> None:
        """Print formatted evaluation report."""
        print("\n" + "="*80)
        print("AGENT EVALUATION REPORT")
        print("="*80)
        
        print(f"\nTotal Queries: {evaluation['total_queries']}")
        print(f"Tool Selection Accuracy: {evaluation['tool_selection_accuracy']:.2%}")
        print(f"Success Rate: {evaluation['success_rate']:.2%}")
        
        print("\n" + "-"*80)
        print("INDIVIDUAL RESULTS")
        print("-"*80)
        
        for i, result in enumerate(evaluation['results'], 1):
            status = "✓" if result.tool_correct else "✗"
            print(f"\n{i}. {status} {result.query}")
            print(f"   Expected: {result.expected_tool}")
            print(f"   Actual: {result.actual_tool or 'None'}")
            
            if result.error:
                print(f"   Error: {result.error}")
            else:
                # Show first 150 chars of response
                response_preview = result.response[:150] + "..." if len(result.response) > 150 else result.response
                print(f"   Response: {response_preview}")
        
        print("\n" + "="*80)

## 7. Initialization

Initialize MCP server and agent orchestrator.

In [11]:
# Initialize MCP server
mcp_server = MCPServerImpl()

# Initialize agent
agent = AgentOrchestrator(
    mcp_server=mcp_server,
    model="mistral",
    max_iterations=5
)

print(f"Registered {len(agent.tools)} tools")
for tool_name in agent.tools.keys():
    print(f"  - {tool_name}")

Registered 2 tools
  - get_current_weather
  - get_latest_news


## 8. Example Usage

Demonstrate agent capabilities with sample queries.

In [12]:
# Example 1: Weather query
print("EXAMPLE 1: Weather Query")

query = "What's the current weather in Paris?"
print(f"\nUser: {query}")

response = agent.process_query(query)
print(f"\nAgent: {response}")

agent.reset()

EXAMPLE 1: Weather Query

User: What's the current weather in Paris?
LLM response:  <TOOL_CALL>
{"name": "get_current_weather", "arguments": {"location": "Paris"}}
</TOOL_CALL>...
Tool result: Tool 'get_current_weather' returned: {
  "location": "Paris, France",
  "temperature_celsius": 15.6,
  "conditions": "Clear sky",
  "humidity_percent": 48,
  "wind_speed_kmh": 8.0,
  "timestamp": "202...
LLM response:  The current weather in Paris is Clear Sky with a temperature of 15.6 degrees Celsius, humidity of 48%, and wind speed of 8 km/h as of April 6th, 2026 at 19:30....

Agent:  The current weather in Paris is Clear Sky with a temperature of 15.6 degrees Celsius, humidity of 48%, and wind speed of 8 km/h as of April 6th, 2026 at 19:30.


In [13]:
# Example 2: News query
print("EXAMPLE 2: News Query")

query = "What are the latest technology news?"
print(f"\nUser: {query}")

response = agent.process_query(query)
print(f"\nAgent: {response}")

agent.reset()

EXAMPLE 2: News Query

User: What are the latest technology news?
LLM response:  <TOOL_CALL>
{"name": "get_latest_news", "arguments": {"category": "technology"}}
</TOOL_CALL>...
Tool result: Tool 'get_latest_news' returned: {
  "category": "technology",
  "count": 5,
  "articles": [
    {
      "title": "How to outsmart fake news in your Facebook feed",
      "summary": "Fake news is actu...
LLM response:  Here are the latest technology news articles as of November 18, 2016:

1. How to outsmart fake news in your Facebook feed - CNN
   Fake news is actually really easy to spot -- if you know how. Consid...

Agent:  Here are the latest technology news articles as of November 18, 2016:

1. How to outsmart fake news in your Facebook feed - CNN
   Fake news is actually really easy to spot -- if you know how. Consider this your New Media Literacy Guide. (Link: http://www.cnn.com/2016/11/18/tech/how-to-spot-fake-misleading-news-trnd/index.html?eref=rss_tech)

2. Revealed: Winners of the 'Osca

In [14]:
# Example 3: Complex query requiring reasoning
print("EXAMPLE 3: Complex Query")

query = "Is it raining in London right now?"
print(f"\nUser: {query}")

response = agent.process_query(query)
print(f"\nAgent: {response}")

agent.reset()

EXAMPLE 3: Complex Query

User: Is it raining in London right now?
LLM response:  <TOOL_CALL>
{"name": "get_current_weather", "arguments": {"location": "London"}}
</TOOL_CALL>...
Tool result: Tool 'get_current_weather' returned: {
  "location": "London, United Kingdom",
  "temperature_celsius": 12.8,
  "conditions": "Clear sky",
  "humidity_percent": 43,
  "wind_speed_kmh": 11.5,
  "timest...
LLM response:  According to the information provided, it is not raining in London right now as the conditions are clear sky....

Agent:  According to the information provided, it is not raining in London right now as the conditions are clear sky.


## 9. Evaluation

Run comprehensive evaluation suite to measure agent performance.

In [15]:
# Initialize evaluator
evaluator = AgentEvaluator(agent)

# Run evaluation
evaluation_results = evaluator.evaluate_all()

# Print report
evaluator.print_report(evaluation_results)

Evaluating: What's the weather like in London?
LLM response:  <TOOL_CALL>
{"name": "get_current_weather", "arguments": {"location": "London"}}
</TOOL_CALL>...
Tool result: Tool 'get_current_weather' returned: {
  "location": "London, United Kingdom",
  "temperature_celsius": 12.8,
  "conditions": "Clear sky",
  "humidity_percent": 43,
  "wind_speed_kmh": 11.5,
  "timest...
LLM response:  The weather in London is currently Clear Sky with a temperature of 12.8 degrees Celsius. The humidity is at 43% and the wind speed is 11.5 km/h as of 7:30 PM local time on April 6, 2026....
Evaluating: Tell me the temperature in Tokyo
LLM response:  <TOOL_CALL>
{"name": "get_current_weather", "arguments": {"location": "Tokyo"}}
</TOOL_CALL>...
Tool result: Tool 'get_current_weather' returned: {
  "location": "Tokyo, Japan",
  "temperature_celsius": 15.3,
  "conditions": "Partly cloudy",
  "humidity_percent": 86,
  "wind_speed_kmh": 8.4,
  "timestamp": "...
LLM response:  The current temperature in Toky

## 10. Architectural Observations

- MCP Server
    - Clean separation: standardized interface, `list_tools()` and `call_tool()` contract
    - Easy extensibility: New tools can be added without modifying agent code
    - JSON Schema: Standard format for tool parameters
- Agent Orchestration
    - LLM-driven: No hardcoded tool selection logic
    - Iterative reasoning: Supports multi-turn tool usage
- Working Memory
    - Token efficiency: Avoids sending full chat history
    - Context preservation: Maintains recent messages and summary
    - Tool result caching: Stores results for potential reuse

## 11. Challenges

- LLM Output Reliability
  - Models may not consistently follow strict output formats (e.g., JSON tool calls)
  - Requires careful prompt engineering and defensive parsing
- Tool Call Parsing
  - Extracting structured data from free-form LLM responses is error-prone
  - Small formatting deviations can break the agent loop
- External API Limitations, unexpected responses
- Error Handling Gaps
  - Failures in tool execution or parsing are only partially handled
  - No retry or fallback strategies
- Prompt Sensitivity
  - System behavior heavily depends on prompt quality
  - Small prompt changes can significantly affect outcomes

## 12. Conclusion

This project demonstrates a from-scratch implementation of an LLM-powered agent capable of reasoning, selecting tools, and generating responses in an iterative loop.

Key takeaways:

- A minimal agent architecture can be built without external frameworks such as LangChain, providing full control and transparency
- A MCP server simplifies tool management and reduces orchestration complexity
- The combination of LLM reasoning + tool execution** enables dynamic, context-aware responses to user queries
- However, building such systems exposes important challenges:
  - reliability of LLM outputs
  - need for robust validation and parsing
  - handling external system dependencies

While not production-ready, this implementation serves as a solid conceptual and practical foundation for:
- understanding agent orchestration patterns
- experimenting with tool-augmented LLM systems
- evolving toward more scalable and secure architectures

Future improvements could include:
- structured output enforcement (e.g., JSON schema / Pydantic)
- retry and fallback mechanisms
- async execution and performance optimization
- enhanced observability and monitoring
- integration with production-grade agent frameworks if needed
